In [1]:
# import libraries
import pandas as pd
import json
import re
import sys
from sklearn.metrics import classification_report as sklearn_classification_report
from seqeval.metrics import classification_report as seqeval_classification_report

# import custom helper functions
sys.path.append("../02_utils/")
from preprocessing import create_regex_pattern
# from classification import tokenize_word_level, text_to_bio
from custom_evaluation import extract_spans, mention_level_evaluation, sentence_level_evaluation

# import the annotations
with open("../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# import the dictionary
group_dictionary_df = pd.read_csv("../01_data/groups_dictionary.csv")

In [ ]:
# function to convert the annotations to bio tags on the word level
def tokenize_word_level(sentence):
    
    # get all words' start and end index
    word_spans = []
    char_idx = 0

    # split sentence via regex which ensures to also split at punctuation
    words = re.findall(r"\w+|'\w+|[^\w\s]", sentence)

    for word in words:
        start_idx = sentence.find(word, char_idx)
        end_idx = start_idx + len(word)
        word_spans.append((start_idx, end_idx))
        char_idx = end_idx
    
    return words, word_spans

def text_to_bio(task):

    # extract sentence and annotations
    sentence = task["sentence"]
    annotations = task["annotations"]

    words, word_spans = tokenize_word_level(sentence)

    # initialize all tags as being O
    bio_tags = ["O"] * len(words)
    
    # loop through the annotations
    for annotation in annotations:
        start_ann, end_ann = annotation["start"], annotation["end"]
        for idx, (start_idx, end_idx) in enumerate(word_spans):
            if start_idx == start_ann:
                bio_tags[idx] = "B-sg"
            elif start_idx > start_ann and end_idx <= end_ann:
                bio_tags[idx] = "I-sg"

    return bio_tags

In [2]:
# add bio tags to the dataset
for task in data:
    bio_tags = text_to_bio(task)
    task["bio_tags"] = bio_tags

# create the regex pattern
combined_regex = create_regex_pattern(group_dictionary_df)

In [3]:
def find_dictionary_matches(sentence, dictionary_regex):
    
    # first tokenize the sentence
    words, word_spans = tokenize_word_level(sentence)

    bio_tags = ["O"] * len(words)

    for match in re.finditer(dictionary_regex, sentence, re.IGNORECASE):
        start_match, end_match = match.span()

        for idx, (start_idx, end_idx) in enumerate(word_spans):
            if start_idx == start_match:
                bio_tags[idx] = "B-sg"
            elif start_idx > start_match and end_idx <= end_match:
                bio_tags[idx] = "I-sg"
    
    return bio_tags

In [4]:
# evaluate on the word level

# store all bio tags in a list
gt_bio_flat = [tag for sent in data for tag in sent["bio_tags"]]
gt_bio_nested = [sent["bio_tags"] for sent in data]

pred_bio_nested = []

for task in data:
    sentence = task["sentence"]
    pred_tags = find_dictionary_matches(sentence, combined_regex)
    pred_bio_nested.append(pred_tags)

# flatten the list
pred_bio_flat = [tag for sent in pred_bio_nested for tag in sent]

# evaluate at the word and at the entity level using seqeval
print("Evaluation at the word level")
print("-"*60)
print(sklearn_classification_report(gt_bio_flat, pred_bio_flat))
print("Evaluation at the entity level with seqeval")
print("-"*60)
print(seqeval_classification_report(gt_bio_nested, pred_bio_nested))

Evaluation at the word level
------------------------------------------------------------
              precision    recall  f1-score   support

        B-sg       0.61      0.64      0.63       597
        I-sg       0.69      0.17      0.27       650
           O       0.98      0.99      0.99     27982

    accuracy                           0.97     29229
   macro avg       0.76      0.60      0.63     29229
weighted avg       0.96      0.97      0.96     29229

Evaluation at the entity level with seqeval
------------------------------------------------------------
              precision    recall  f1-score   support

          sg       0.53      0.56      0.54       597

   micro avg       0.53      0.56      0.54       597
   macro avg       0.53      0.56      0.54       597
weighted avg       0.53      0.56      0.54       597



In [5]:
# evaluate on the entity level
all_true_spans = []
all_predicted_spans = []

for idx in range(len(gt_bio_nested)):

    # get the spans
    all_true_spans.append(extract_spans(gt_bio_nested[idx]))
    all_predicted_spans.append(extract_spans(pred_bio_nested[idx]))
 
# apply cross-span evaluation
mention_level_evaluation(all_true_spans=all_true_spans, all_predicted_spans=all_predicted_spans)

{'precision': 0.6784260515603799,
 'recall': 0.5506488762255384,
 'f1': 0.5789194636209561}

In [6]:
sentence_level_evaluation(all_true_spans=gt_bio_nested, all_predicted_spans=pred_bio_nested)

{'precision': 0.834, 'recall': 0.9084967320261438, 'f1': 0.8696558915537017}